# 02 — Carga da Camada Bronze (Batch)

Este notebook lê as origens criadas em `01_criacao_origens.py` e persiste os dados
**brutos** na camada Bronze com metadados de rastreabilidade.

**Características da Bronze**:
- Dados ingeridos sem transformações significativas
- Histórico completo preservado (append-only)
- Metadados de ingestão em todas as tabelas (`_data_ingestao_bronze`, `_fonte`)
- JSON de eventos CDC mantido como coluna parseada para auditoria

**Próximo passo**: executar `03_carga_camada_silver.py`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window

# Verifica pré-requisitos
tabelas_origem = [
    "origens.tc02_uf",
    "origens.tc02_meta_brasil",
    "origens.tc02_cdc_meta_uf_eventos_json",
    "origens.tc02_indicador_municipio_json",
    "origens.tc02_municipio",
]

for tabela in tabelas_origem:
    try:
        spark.read.table(tabela).limit(1).count()
    except Exception as err:
        raise ValueError(
            f"Tabela '{tabela}' não encontrada. Execute primeiro 01_criacao_origens.py."
        ) from err

print("Pré-requisitos verificados. Iniciando carga Bronze...")

## Bronze 1: Dimensão UF

In [0]:
df_bronze_uf = (
    spark.read.table("origens.tc02_uf")
    .withColumn("_data_ingestao_bronze", F.current_timestamp())
    .withColumn("_fonte", F.lit("origens.tc02_uf"))
)

display(df_bronze_uf)

## Bronze 2: Meta Nacional (Brasil)

In [0]:
df_bronze_meta_brasil = (
    spark.read.table("origens.tc02_meta_brasil")
    .withColumn("_data_ingestao_bronze", F.current_timestamp())
    .withColumn("_fonte", F.lit("origens.tc02_meta_brasil"))
)

display(df_bronze_meta_brasil)

## Bronze 3: Meta por UF (via CDC)

Lê os eventos CDC brutos, parseia o JSON e resolve o estado mais recente
de cada `(sigla_uf, ano)` usando Window function — preservando o histórico de revisões.

In [0]:
schema_cdc_meta_uf = T.StructType([
    T.StructField("Op", T.StringType(), True),
    T.StructField("tabela", T.StringType(), True),
    T.StructField("commit_timestamp", T.StringType(), True),
    T.StructField("data", T.StructType([
        T.StructField("sigla_uf", T.StringType(), True),
        T.StructField("ano", T.IntegerType(), True),
        T.StructField("meta_uf", T.DoubleType(), True),
        T.StructField("versao", T.IntegerType(), True),
    ]), True),
])

df_raw_cdc = spark.read.table("origens.tc02_cdc_meta_uf_eventos_json")

df_cdc_parsed = (
    df_raw_cdc
    .select(F.from_json(F.col("json_evento"), schema_cdc_meta_uf).alias("ev"))
    .select("ev.*")
)

janela_cdc = Window.partitionBy("data.sigla_uf", "data.ano").orderBy(F.col("commit_timestamp").desc())

df_bronze_meta_uf = (
    df_cdc_parsed
    .filter(F.col("Op").isin("I", "U"))
    .withColumn("rank_cdc", F.row_number().over(janela_cdc))
    .filter(F.col("rank_cdc") == 1)
    .select(
        F.col("data.sigla_uf").alias("sigla_uf"),
        F.col("data.ano").alias("ano"),
        F.col("data.meta_uf").alias("meta_uf"),
        F.col("data.versao").alias("versao_meta"),
        F.col("commit_timestamp").alias("ultima_atualizacao_meta"),
    )
    .withColumn("_data_ingestao_bronze", F.current_timestamp())
    .withColumn("_fonte", F.lit("origens.tc02_cdc_meta_uf_eventos_json"))
)

print(f"Bronze Meta UF: {df_bronze_meta_uf.count()} registros")
display(df_bronze_meta_uf.orderBy("sigla_uf"))

## Bronze 4: Indicador por Município (via Arquivo JSON)

Parseia os registros JSON e mantém dados brutos com metadados de ingestão.
Os registros inválidos (duplicatas, nulos) são preservados — a Silver os tratará.

In [0]:
schema_indicador = T.StructType([
    T.StructField("id_municipio", T.IntegerType(), True),
    T.StructField("nome_municipio", T.StringType(), True),
    T.StructField("sigla_uf", T.StringType(), True),
    T.StructField("ano", T.IntegerType(), True),
    T.StructField("total_alunos_2o_ano", T.IntegerType(), True),
    T.StructField("alunos_alfabetizados", T.IntegerType(), True),
    T.StructField("indicador_crianca_alfabetizada", T.DoubleType(), True),
    T.StructField("ponto_corte_saeb", T.IntegerType(), True),
    T.StructField("fonte", T.StringType(), True),
])

df_bronze_indicador = (
    spark.read.table("origens.tc02_indicador_municipio_json")
    .select(F.from_json(F.col("json_linha"), schema_indicador).alias("r"))
    .select("r.*")
    .withColumn("_data_ingestao_bronze", F.current_timestamp())
    .withColumn("_fonte", F.lit("origens.tc02_indicador_municipio_json"))
)

print(f"Bronze Indicador Município: {df_bronze_indicador.count()} registros (inclui duplicatas/inválidos)")
display(df_bronze_indicador.orderBy("sigla_uf", "ano").limit(20))

## Bronze 5: Dimensão Município

In [0]:
df_bronze_municipio = (
    spark.read.table("origens.tc02_municipio")
    .withColumn("_data_ingestao_bronze", F.current_timestamp())
    .withColumn("_fonte", F.lit("origens.tc02_municipio"))
)

print(f"Bronze Município: {df_bronze_municipio.count()} registros")
display(df_bronze_municipio)

## Persistência na Camada Bronze

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

tabelas_bronze = {
    "bronze.tc02_uf_raw":                  df_bronze_uf,
    "bronze.tc02_meta_brasil_raw":         df_bronze_meta_brasil,
    "bronze.tc02_meta_uf_raw":             df_bronze_meta_uf,
    "bronze.tc02_indicador_municipio_raw": df_bronze_indicador,
    "bronze.tc02_municipio_raw":           df_bronze_municipio,
}

for nome, df in tabelas_bronze.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome)
    )

print("Tabelas Bronze criadas com sucesso:")
for nome in tabelas_bronze:
    print(f"  - {nome} => {spark.read.table(nome).count()} linhas")

print("\nPróximo passo: executar 03_carga_camada_silver.py")